# **Kitchen Usage Workflow**

The kitchen monitoring module integrated into DepaYT combines contextual analysis and computer vision techniques to support cleanliness tracking in shared apartment environments.

**1. Start Kitchen Usage**: The workflow begins when a user presses the `Start Usage` button inside the application before using the kitchen. At this stage, the system starts a cooking session associated with the current user.

 **2. Stop Kitchen Usage**: After finishing the cooking activity, the user presses the `Stop Usage` button. Once the session ends, the application displays a contextual questionnaire related to the cooking activity performed.

The questionnaire includes information such as:

- type of food prepared,
- cooking method,
- oil usage,
- spill occurrence,
- number of utensils used,
- number of people served.

**3. Contextual Dirtiness Prediction**: The answers provided by the user are converted into a structured tabular input and processed by a Multilayer Perceptron (MLP) model trained on contextual cooking data.

The MLP predicts the expected kitchen dirtiness level, classified into one of the following categories:

- low,
- medium,
- high.

Based on the predicted level, the system generates personalized cleaning recommendations to guide the user in properly cleaning the kitchen after use.

 **4. Upload Final Kitchen Image**: After receiving the recommendation, the user uploads a final image of the induction kitchen surface. The image is processed by a Convolutional Neural Network (CNN) trained for binary kitchen cleanliness classification.

The CNN predicts whether the kitchen surface is:
- clean,
- dirty.

**5. Save and Share Kitchen Status**
The final cleanliness result, along with the contextual session information, is stored inside the application database.
This information can later be visualized by all roommates, allowing users to monitor kitchen usage history, cleanliness status, and shared kitchen conditions over time.

The proposed workflow combines:
- contextual reasoning through the MLP model,
- visual cleanliness verification through the CNN model,
- collaborative monitoring inside a shared apartment environment.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib
from tensorflow.keras.models import load_model


In [ ]:
preprocessor = joblib.load(
    "Text_Questions/data/preprocessor.joblib"
)

label_encoder = joblib.load(
    "Text_Questions/data/label_encoder.joblib"
)


mlp_model = load_model(
    r"Text_Questions\results\kitchen_context_mlp.keras"
)

In [ ]:
recommendations = {

    "bajo":
        (
            "Se detectó un nivel de suciedad esperado bajo. "
            "La actividad realizada probablemente generó pocos residuos sobre la superficie de inducción. "
            "Aun así, se recomienda limpiar ligeramente la cocina después de terminar para evitar acumulación progresiva de grasa, polvo o manchas pequeñas que pueden adherirse con el tiempo. "
            "Puedes utilizar un trapo húmedo, papel absorbente o una pequeña cantidad de desinfectante suave para limpiar la superficie. "
            "También es recomendable verificar que no hayan quedado gotas de agua, residuos de comida o utensilios olvidados alrededor del área de cocina. "
            "Mantener una limpieza constante ayuda a conservar el buen estado de la cocina compartida y facilita futuras sesiones de uso."
        ),

    "medio":
        (
            "Se detectó un nivel de suciedad esperado medio. "
            "La preparación realizada pudo generar residuos visibles como manchas de aceite, restos de alimentos, pequeñas salpicaduras o residuos adheridos alrededor de la superficie de inducción. "
            "Se recomienda realizar una limpieza moderada antes de finalizar la sesión utilizando un trapo húmedo, papel absorbente o productos suaves de limpieza para cocina. "
            "Presta especial atención a las esquinas, bordes y zonas cercanas a las ollas o sartenes donde suelen acumularse residuos de grasa o líquidos. "
            "En caso de derrames, es recomendable limpiarlos inmediatamente para evitar manchas persistentes o superficies pegajosas. "
            "Recuerda también organizar los utensilios utilizados y verificar que el espacio quede listo para el siguiente usuario."
        ),

    "alto":
        (
            "Se detectó un nivel de suciedad esperado alto. "
            "La actividad de cocina realizada probablemente involucró preparación intensiva, uso de aceite, frituras, salsas o múltiples utensilios, lo que puede generar acumulación considerable de grasa, residuos y derrames sobre la superficie de inducción. "
            "Después de cocinar, se recomienda realizar una limpieza profunda de toda el área utilizada. "
            "Puedes ayudarte de un trapo húmedo, papel absorbente, esponjas suaves o desinfectante especializado para cocina con el fin de remover residuos adheridos, manchas de grasa y salpicaduras difíciles. "
            "En caso de derrames de líquidos o salsas, es importante secarlos y limpiarlos inmediatamente para evitar manchas permanentes, malos olores o acumulación de suciedad. "
            "También se recomienda revisar los alrededores de la cocina, incluyendo utensilios, recipientes y superficies cercanas que pudieron verse afectadas durante la preparación. "
            "Mantener la cocina limpia después de una sesión de uso intensivo contribuye a un ambiente compartido más higiénico, seguro y agradable para todos los usuarios."
        )
}

In [8]:
user_input = pd.DataFrame([{
    "tipo_comida": "pollo",
    "metodo_coccion": "frito",
    "uso_aceite": "si",
    "hubo_derrame": "si",
    "cantidad_utensilios": "muchos",
    "cantidad_personas": "3_o_mas"
}])

In [14]:
def predict_kitchen_dirtiness(user_input_df):
    """
    Predicts expected dirtiness level from
    contextual cooking information.

    Parameters:
        user_input_df (pd.DataFrame)

    Returns:
        expected_level
        recommendation
        probabilities
    """
    user_input_df = user_input_df.copy()

    binary_mapping = {
        "si": 1,
        "no": 0
    }

    binary_columns = [
        "uso_aceite",
        "hubo_derrame"
    ]

    for col in binary_columns:
        user_input_df[col] = (
            user_input_df[col]
            .map(binary_mapping)
        )

    # PREPROCESS
    X_user = preprocessor.transform(
        user_input_df
    )

    # PREDICT
    y_probs = mlp_model.predict(
        X_user
    )

    y_pred = np.argmax(
        y_probs,
        axis=1
    )

    # DECODE LABEL
    expected_level = (
        label_encoder
        .inverse_transform(y_pred)[0]
    )

    # GET RECOMMENDATION
    recommendation = recommendations[
        expected_level
    ]

    return {

        "expected_level":
            expected_level,

        "recommendation":
            recommendation,

        "probabilities":
            y_probs[0]
    }




# **Sample Data to test recomendations**

In [15]:
example_input = pd.DataFrame([{
    "tipo_comida": "pollo",
    "metodo_coccion": "frito",
    "uso_aceite": "si",
    "hubo_derrame": "si",
    "cantidad_utensilios": "muchos",
    "cantidad_personas": "3_o_mas"
}])


result = predict_kitchen_dirtiness(
    example_input
)

print("\n========== PREDICTION ==========\n")
print(
    f"\nProbabilities:\n"
    f"{result['probabilities']}"
)

print(
    f"Expected Dirtiness Level: "
    f"{result['expected_level']}"
)

print(
    f"\nRecommendation:\n"
    f"{result['recommendation']}"
)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step

========== PREDICTION ==========


Probabilities:
[9.9998927e-01 2.7074474e-16 1.0732231e-05]
Expected Dirtiness Level: alto

Recommendation:
Se detectó un nivel de suciedad esperado alto. La actividad de cocina realizada probablemente involucró preparación intensiva, uso de aceite, frituras, salsas o múltiples utensilios, lo que puede generar acumulación considerable de grasa, residuos y derrames sobre la superficie de inducción. Después de cocinar, se recomienda realizar una limpieza profunda de toda el área utilizada. Puedes ayudarte de un trapo húmedo, papel absorbente, esponjas suaves o desinfectante especializado para cocina con el fin de remover residuos adheridos, manchas de grasa y salpicaduras difíciles. En caso de derrames de líquidos o salsas, es importante secarlos y limpiarlos inmediatamente para evitar manchas permanentes, malos olores o acumulación de suciedad. También se recomienda revisar los alrededores de la cocina, incluyendo 

In [16]:
example_input = pd.DataFrame([{
    "tipo_comida": "pollo",
    "metodo_coccion": "recalentado",
    "uso_aceite": "no",
    "hubo_derrame": "no",
    "cantidad_utensilios": "pocos",
    "cantidad_personas": "1_o_2"
}])


result = predict_kitchen_dirtiness(
    example_input
)

print("\n========== PREDICTION ==========\n")
print(
    f"\nProbabilities:\n"
    f"{result['probabilities']}"
)

print(
    f"Expected Dirtiness Level: "
    f"{result['expected_level']}"
)

print(
    f"\nRecommendation:\n"
    f"{result['recommendation']}"
)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step

========== PREDICTION ==========


Probabilities:
[3.0560368e-09 9.9932969e-01 6.7036878e-04]
Expected Dirtiness Level: bajo

Recommendation:
Se detectó un nivel de suciedad esperado bajo. La actividad realizada probablemente generó pocos residuos sobre la superficie de inducción. Aun así, se recomienda limpiar ligeramente la cocina después de terminar para evitar acumulación progresiva de grasa, polvo o manchas pequeñas que pueden adherirse con el tiempo. Puedes utilizar un trapo húmedo, papel absorbente o una pequeña cantidad de desinfectante suave para limpiar la superficie. También es recomendable verificar que no hayan quedado gotas de agua, residuos de comida o utensilios olvidados alrededor del área de cocina. Mantener una limpieza constante ayuda a conservar el buen estado de la cocina compartida y facilita futuras sesiones de uso.


In [18]:
example_input = pd.DataFrame([{
    "tipo_comida": "pasta",
    "metodo_coccion": "salteado",
    "uso_aceite": "si",
    "hubo_derrame": "no",
    "cantidad_utensilios": "medios",
    "cantidad_personas": "1_o_2"
}])


result = predict_kitchen_dirtiness(
    example_input
)

print("\n========== PREDICTION ==========\n")

print(
    f"\nProbabilities:\n"
    f"{result['probabilities']}"
)

print(
    f"Expected Dirtiness Level: "
    f"{result['expected_level']}"
)

print(
    f"\nRecommendation:\n"
    f"{result['recommendation']}"
)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step

========== PREDICTION ==========


Probabilities:
[8.1566805e-03 6.0317578e-04 9.9124008e-01]
Expected Dirtiness Level: medio

Recommendation:
Se detectó un nivel de suciedad esperado medio. La preparación realizada pudo generar residuos visibles como manchas de aceite, restos de alimentos, pequeñas salpicaduras o residuos adheridos alrededor de la superficie de inducción. Se recomienda realizar una limpieza moderada antes de finalizar la sesión utilizando un trapo húmedo, papel absorbente o productos suaves de limpieza para cocina. Presta especial atención a las esquinas, bordes y zonas cercanas a las ollas o sartenes donde suelen acumularse residuos de grasa o líquidos. En caso de derrames, es recomendable limpiarlos inmediatamente para evitar manchas persistentes o superficies pegajosas. Recuerda también organizar los utensilios utilizados y verificar que el espacio quede listo para el siguiente usuario.
